In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Shared Drive folder. Teammate: add a SHORTCUT to this folder in My Drive
# so this exact path resolves for you too.
DRIVE = Path('/content/drive/MyDrive/m5_project')
RAW_DRIVE = DRIVE / 'raw'          # the zip lives here, shared
LOCAL = Path('/content/m5')        # extracted CSVs, private to your VM

for p in [RAW_DRIVE, DRIVE/'processed', DRIVE/'models', DRIVE/'reports', LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

COMPETITION = 'm5-forecasting-accuracy'
ZIP_NAME = f'{COMPETITION}.zip'
print('Drive ready:', DRIVE)

Mounted at /content/drive
Drive ready: /content/drive/MyDrive/m5_project


In [ ]:
!pip install -q --upgrade kaggle

from google.colab import userdata
from pathlib import Path
import json, os

creds = json.loads(userdata.get('Snehil-kaggle'))
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY'] = creds['key']
print('Kaggle auth set for:', creds['username'])   # username only, never the key

Kaggle auth set for: snehilraut


In [ ]:
import zipfile

zip_path = RAW_DRIVE / ZIP_NAME

if zip_path.exists():
    print('Zip already in Drive, skipping download.')
else:
    !kaggle competitions download {COMPETITION} -p {RAW_DRIVE}
    print('Downloaded.')

with zipfile.ZipFile(zip_path) as z:
    z.extractall(LOCAL)

print(sorted(p.name for p in LOCAL.glob('*.csv')))

100% 45.8M/45.8M [00:00<00:00, 67.2MB/s]

Downloaded.
['calendar.csv', 'sales_train_evaluation.csv', 'sales_train_validation.csv', 'sample_submission.csv', 'sell_prices.csv']


In [ ]:
import pandas as pd

calendar   = pd.read_csv(LOCAL / 'calendar.csv')
prices     = pd.read_csv(LOCAL / 'sell_prices.csv')
sales      = pd.read_csv(LOCAL / 'sales_train_evaluation.csv')   # d_1 .. d_1941
submission = pd.read_csv(LOCAL / 'sample_submission.csv')

for name, df in [('calendar', calendar), ('sell_prices', prices),
                 ('sales_eval', sales), ('submission', submission)]:
    mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f'{name:<12} {df.shape}   {mb:8.1f} MB')

calendar     (1969, 14)        0.7 MB
sell_prices  (6841121, 4)      853.1 MB
sales_eval   (30490, 1947)      461.9 MB
submission   (60980, 29)       17.5 MB


In [ ]:
import re

d_cols = [c for c in sales.columns if re.fullmatch(r'd_\d+', c)]
print('day columns:', d_cols[0], '..', d_cols[-1], f'({len(d_cols)} days)')
print('date range :', calendar.date.min(), '..', calendar.date.max())
print('series     :', sales.id.nunique())
print('per store  :')
print(sales.store_id.value_counts().to_string())

day columns: d_1 .. d_1941 (1941 days)
date range : 2011-01-29 .. 2016-06-19
series     : 30490
per store  :
store_id
CA_1    3049
CA_2    3049
CA_3    3049
CA_4    3049
TX_1    3049
TX_2    3049
TX_3    3049
WI_1    3049
WI_2    3049
WI_3    3049
